# Week 2 - SQL Exercises: JOINs Challenge

## Practice Problems - INNER JOIN, LEFT JOIN, Self-Join, Multi-Table Join, Orphan Detection, Aggregation After Joins

This notebook contains 6 problems of increasing difficulty. Each problem asks you to write a SQL query against the `company` schema in your PostgreSQL database.

### How to use this notebook:
1. Make sure your PostgreSQL database is running (Docker container with `week2_db`).
2. Run the connection helper cell below to enable `run_query()`.
3. Write your SQL query in each empty code cell.
4. If you get stuck, scroll down to the **HINT** cell (but try first!).
5. After attempting the problem, scroll further to the **SOLUTION** to check your answer.

### Schema reference:
All tables are in the `company` schema. Key tables for these exercises:
- **departments**: `dept_id, dept_name, location, budget`
- **employees**: `emp_id, first_name, last_name, email, department_id, salary, hire_date, manager_id, is_active`
- **products**: `product_id, product_name, category, unit_price`
- **sales**: `sale_id, employee_id, product_id, quantity, sale_date, region`

---
### Connection Helper
Run this cell once to set up the `run_query()` helper function.

In [ ]:
import psycopg2
import pandas as pd
from IPython.display import display, HTML

def run_query(query):
    """Execute a SQL query against the week2_db PostgreSQL database and return a DataFrame."""
    conn = psycopg2.connect(
        host="localhost",
        port=5432,
        dbname="week2_db",
        user="student",
        password="student123"
    )
    try:
        df = pd.read_sql_query(query, conn)
        display(df)
        return df
    except Exception as e:
        print(f"Error: {e}")
        return None
    finally:
        conn.close()

---

## Problem 1: Employee Directory with Departments (LEFT JOIN)

**Business Question:**  
HR needs a complete employee directory showing every employee alongside their department name. Importantly, some employees may not yet be assigned to a department (their `department_id` is NULL). Those employees must still appear in the results with `NULL` shown as their department name. Display the employee's first name, last name, email, and the department name. Order alphabetically by last name, then first name.

In [ ]:
# Write your query here
query_1 = """

"""
run_query(query_1)

> **HINT** - scroll down only if stuck  
> You need *all* employees, even those without a matching department. That means `employees` should be the left table in a `LEFT JOIN` to `departments` on `department_id = dept_id`. Use `COALESCE(dept_name, 'Unassigned')` if you want a friendly label instead of NULL.

<details>
<summary><strong>SOLUTION - Problem 1</strong> (click to reveal)</summary>

```sql
SELECT e.first_name,
       e.last_name,
       e.email,
       d.dept_name
FROM company.employees e
LEFT JOIN company.departments d
       ON e.department_id = d.dept_id
ORDER BY e.last_name, e.first_name;
```

**Explanation:** A `LEFT JOIN` keeps every row from the left table (`employees`) regardless of whether a match exists in the right table (`departments`). The employee Paula Morris has `department_id = NULL`, so her `dept_name` will appear as NULL in the result. This is the correct behavior for a LEFT JOIN.

**Optional improvement:** Use `COALESCE(d.dept_name, 'Unassigned') AS dept_name` to replace NULL with a readable label.

**Key difference from INNER JOIN:** If we used `INNER JOIN` here, Paula Morris would be excluded from the results entirely because her `department_id` does not match any `dept_id`.
</details>

---

## Problem 2: Employees Who Have NEVER Made a Sale (LEFT JOIN + IS NULL)

**Business Question:**  
The Sales VP wants to identify employees who are on the payroll but have never recorded a single sale. Find all employees who do not appear in the `sales` table at all. Show their employee ID, first name, last name, and department ID. Order by employee ID.

In [ ]:
# Write your query here
query_2 = """

"""
run_query(query_2)

> **HINT** - scroll down only if stuck  
> Start with `employees` and `LEFT JOIN` to `sales` on `emp_id = employee_id`. Then filter the result to only rows where `sale_id IS NULL`. Those are employees with no matching sales record.

<details>
<summary><strong>SOLUTION - Problem 2</strong> (click to reveal)</summary>

```sql
SELECT e.emp_id,
       e.first_name,
       e.last_name,
       e.department_id
FROM company.employees e
LEFT JOIN company.sales s
       ON e.emp_id = s.employee_id
WHERE s.sale_id IS NULL
ORDER BY e.emp_id;
```

**Explanation:** The `LEFT JOIN` produces all employees, attaching sale records where they exist. For employees who have never made a sale, all columns from the `sales` table will be NULL. By filtering `WHERE s.sale_id IS NULL`, we keep only those orphan employee rows. This is a classic "anti-join" pattern in SQL.

**Expected results:** This will include department directors and managers from non-sales departments (Engineering, HR, Finance, Marketing, Operations) who never appear in the sales table, plus any sales staff who were hired but haven't closed a deal yet.
</details>

---

## Problem 3: Employee and Manager Names (Self-Join)

**Business Question:**  
For an organizational chart visualization, show each employee alongside their direct manager's full name. Display the employee's first and last name, and their manager's first and last name (as `manager_first_name` and `manager_last_name`). For employees at the top of the hierarchy (no manager), show `NULL` for the manager's name. Order by the employee's last name, then first name.

In [ ]:
# Write your query here
query_3 = """

"""
run_query(query_3)

> **HINT** - scroll down only if stuck  
> A self-join means joining a table to itself. Alias `employees` as `e` (the employee) and again as `m` (the manager). Join on `e.manager_id = m.emp_id`. Use a `LEFT JOIN` so that top-level managers (with `manager_id = NULL`) still appear in the results.

<details>
<summary><strong>SOLUTION - Problem 3</strong> (click to reveal)</summary>

```sql
SELECT e.first_name,
       e.last_name,
       m.first_name AS manager_first_name,
       m.last_name  AS manager_last_name
FROM company.employees e
LEFT JOIN company.employees m
       ON e.manager_id = m.emp_id
ORDER BY e.last_name, e.first_name;
```

**Explanation:** A self-join treats the same table as two different roles. Here, `e` represents the subordinate and `m` represents the manager. The join condition `e.manager_id = m.emp_id` links each employee to their manager. We use `LEFT JOIN` because the six department directors (Alice Chen, Iris Taylor, Nathan Harris, Uma King, Amy Adams, James Campbell) plus Paula Morris all have `manager_id = NULL` and must still appear in the output.

**Hierarchy levels in this data:**
- Level 1 (no manager): Directors and Paula Morris
- Level 2: Managers (report to directors)
- Level 3: Individual contributors (report to managers)
</details>

---

## Problem 4: Product Sales Summary (JOIN + GROUP BY)

**Business Question:**  
The product team wants a sales summary for each product. For every product that has been sold at least once, show the product name, the total quantity sold across all sales, and the total revenue (calculated as the sum of `quantity * unit_price`). Round the total revenue to 2 decimal places. Order by total revenue from highest to lowest.

In [ ]:
# Write your query here
query_4 = """

"""
run_query(query_4)

> **HINT** - scroll down only if stuck  
> Join `products` to `sales` on `product_id`. Then use `SUM(quantity)` for total quantity and `SUM(quantity * unit_price)` for total revenue. Group by `product_name`. Use `ROUND(..., 2)` for the revenue column. An `INNER JOIN` is appropriate here since we only want products that have been sold.

<details>
<summary><strong>SOLUTION - Problem 4</strong> (click to reveal)</summary>

```sql
SELECT p.product_name,
       SUM(s.quantity)                    AS total_quantity,
       ROUND(SUM(s.quantity * p.unit_price), 2) AS total_revenue
FROM company.products p
INNER JOIN company.sales s
       ON p.product_id = s.product_id
GROUP BY p.product_name
ORDER BY total_revenue DESC;
```

**Explanation:** The `INNER JOIN` matches each sale to its product. The `SUM(quantity * unit_price)` calculates revenue per sale row (since `unit_price` comes from the products table, it is the same for all rows of a given product, so `quantity * unit_price` gives the revenue for that line item). Grouping by `product_name` aggregates all sales of the same product together.

**Important note:** The product "Legacy System Module" (product_id = 17) will NOT appear in these results because it has zero sales. An `INNER JOIN` excludes it. If you wanted to include it with zero values, you would use a `LEFT JOIN` instead and wrap the aggregates with `COALESCE`.
</details>

---

## Problem 5: Departments with Zero Employees (LEFT JOIN + Orphan Detection)

**Business Question:**  
Executive leadership is reviewing department staffing. Find any departments that currently have zero employees assigned to them. Show the department name and location. (Note: based on the current dataset, this may return zero rows, but writing the correct query pattern is the goal.)

In [ ]:
# Write your query here
query_5 = """

"""
run_query(query_5)

> **HINT** - scroll down only if stuck  
> Start with `departments` as the left table and `LEFT JOIN` to `employees` on `dept_id = department_id`. Then filter `WHERE emp_id IS NULL` to find departments with no matching employees. This is the same "anti-join" pattern as Problem 2, but with departments as the left table.

<details>
<summary><strong>SOLUTION - Problem 5</strong> (click to reveal)</summary>

```sql
SELECT d.dept_name,
       d.location
FROM company.departments d
LEFT JOIN company.employees e
       ON d.dept_id = e.department_id
WHERE e.emp_id IS NULL;
```

**Explanation:** This query follows the same anti-join pattern as Problem 2. We keep all departments from the left side, then filter to only those where no employee matched (i.e., `emp_id IS NULL`).

**Expected result:** This query returns **zero rows** with the current dataset, because all 6 departments have at least one employee assigned. However, this is the correct and standard pattern for finding orphan records. If HR were to remove all employees from a department in the future, this query would immediately surface it.

**When this pattern is useful:** In real-world scenarios, this anti-join pattern is commonly used to find:
- Departments with no staff
- Products with no sales
- Customers with no orders
- Categories with no products
</details>

---

## Problem 6: Full Sale Details with Revenue (Multi-Table Join: 4 Tables)

**Business Question:**  
The Finance team needs a comprehensive sales report. For each sale, show the employee's full name (first + last name combined), the department name, the product name, the quantity sold, and the calculated revenue (`quantity * unit_price`). Order the results by revenue from highest to lowest and limit to the top 20 rows.

This query requires joining **four tables**: `sales`, `employees`, `departments`, and `products`.

In [ ]:
# Write your query here
query_6 = """

"""
run_query(query_6)

> **HINT** - scroll down only if stuck  
> Start from the `sales` table and chain `INNER JOIN`s to the other tables: `sales -> employees` (on `employee_id = emp_id`), `employees -> departments` (on `department_id = dept_id`), and `sales -> products` (on `product_id = product_id`). Use `CONCAT(e.first_name, ' ', e.last_name)` for the full name. Calculate revenue as `(s.quantity * p.unit_price)`. End with `ORDER BY revenue DESC LIMIT 20`.

<details>
<summary><strong>SOLUTION - Problem 6</strong> (click to reveal)</summary>

```sql
SELECT CONCAT(e.first_name, ' ', e.last_name) AS employee_name,
       d.dept_name,
       p.product_name,
       s.quantity,
       p.unit_price,
       ROUND(s.quantity * p.unit_price, 2) AS revenue
FROM company.sales s
INNER JOIN company.employees e
       ON s.employee_id = e.emp_id
INNER JOIN company.departments d
       ON e.department_id = d.dept_id
INNER JOIN company.products p
       ON s.product_id = p.product_id
ORDER BY revenue DESC
LIMIT 20;
```

**Explanation:** This query chains three `INNER JOIN`s across four tables:

1. **sales -> employees**: Links each sale to the employee who made it.
2. **employees -> departments**: Looks up the employee's department.
3. **sales -> products**: Looks up the product details (including unit price).

The `CONCAT` function merges first and last name into a single readable column. Revenue is calculated as `quantity * unit_price` and rounded to 2 decimal places for currency display. Ordering by `revenue DESC` puts the biggest sales at the top, and `LIMIT 20` restricts the output.

**Expected top results:** The highest-revenue individual sales will likely be high-priced Hardware products (like Server Rack X200 at $5,499.99 or Load Balancer LB-500 at $3,499.99) sold in larger quantities.

**Chain join pattern:** The key to multi-table joins is building a clear chain where each join connects through a foreign key relationship. Start from your "fact" table (sales) and branch out to "dimension" tables (employees, products), then from employees to departments.
</details>

---

## Summary of Concepts Covered

| Problem | Key Concepts |
|---------|-------------|
| 1 | `LEFT JOIN` - keep all employees including unassigned |
| 2 | `LEFT JOIN` + `IS NULL` - anti-join to find employees with no sales |
| 3 | Self-join - employee table joined to itself for manager hierarchy |
| 4 | `INNER JOIN` + `GROUP BY` + `SUM()` - product sales aggregation |
| 5 | `LEFT JOIN` + `IS NULL` - anti-join to find empty departments |
| 6 | Multi-table `INNER JOIN` (4 tables) + calculated columns + `ORDER BY` + `LIMIT` |

**Key patterns to remember:**
- **LEFT JOIN** keeps all rows from the left table, filling NULLs where no match exists.
- **Anti-join pattern**: `LEFT JOIN` + `WHERE right_table.pk IS NULL` finds orphan records.
- **Self-join**: A table joined to itself using different aliases for different roles.
- **Multi-table joins**: Chain joins through foreign key relationships starting from your fact table.
- **Aggregation after joins**: `GROUP BY` columns must match what you select (or use aliases in some databases).

**Next steps:** Try modifying these queries - for example, add date filters, change LEFT JOINs to INNER JOINs and compare results, or add HAVING clauses to Problem 4 to find products with total revenue above a threshold.